# Historical forecast study

What the model would have said before each storm arrived, read against how
early it would have said it.

Two figures:

1. **Ensemble map** — one panel per forecast cycle for a single storm, the
   ensemble's tracks over the ADM1 units the model predicted at that cycle.
2. **Precision and recall against time to landfall** — both impact thresholds,
   the two-stage model against the windspeed-exposed baseline, ending at the
   score each reaches once the observed wind field is known.

The ensemble is summarised by its median with the quartiles as a band. That
band is the spread the forecast carried, not a confidence interval on the
metric.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import OUTPUT_DIR
from src.evaluation.forecast_metrics import (ADM1_KEYS, LEAD_EDGES,
                                             build_study_table,
                                             metrics_by_lead, post_landfall)
from visualization.lib import RUN_SUFFIX

#: The forecast centre the study was collected from.
PROVIDER = "rksl"

#: The storm the ensemble map is drawn for. Eight cycles, a full 26-member
#: ensemble and units that reached complete impact, so the map has something
#: to show at every panel.
MAP_EVENT = "2019-0573-PHL"

#: Colours follow the published figure: the model warm, the baseline cool.
COLOUR = {"xgb": "#E24916", "wind": "#AEC6CF"}
MARKER = {"xgb": "s", "wind": "o"}
LABEL = {"xgb": "2-stg-XGBoost", "wind": "windspeed-exposed"}

ROOT = OUTPUT_DIR / "forecast_study" / PROVIDER
FIGDIR = Path.cwd() / f"figs{RUN_SUFFIX}"
FIGDIR.mkdir(exist_ok=True)
print(f"provider {PROVIDER}; run suffix {RUN_SUFFIX!r}; figures -> {FIGDIR}")

## 1. The study, summarised per unit and cycle

The study's grid predictions run to hundreds of millions of rows, so they are
reduced once to one row per (ADM1 unit, forecast cycle) and cached. Delete the
cached file to rebuild it.

In [ ]:
cached = ROOT / "adm1_by_cycle.parquet"
if cached.exists():
    table = pd.read_parquet(cached)
    print(f"read {len(table):,} rows from {cached.name}")
else:
    table = build_study_table(ROOT / "events", cached)

for column in ADM1_KEYS:
    table[column] = table[column].astype(str)

print(f"{table['DisNo.'].nunique()} events, {table['cycle'].nunique()} cycles, "
      f"{len(table):,} (unit, cycle) rows")
print(f"lead time {table['lead_hours'].min():.0f} to "
      f"{table['lead_hours'].max():.0f} h, "
      f"members per cycle {int(table['members'].min())}"
      f"-{int(table['members'].max())}")
table.head(3)

## 2. Skill against lead time

Cycles are binned by how long before arrival they were issued. The last bin's
edge is where the archive thins out; cycles issued earlier than that are
dropped rather than piled into a final bin that would not mean the same thing.

The post-landfall point is the same two models scored on the observed wind
field, restricted to the units this study covers, so it answers what is lost
by forecasting rather than what is lost by looking at different places.

In [ ]:
metrics = metrics_by_lead(table)

model_output = OUTPUT_DIR / "model_output"
loocv = pd.concat(
    [pd.read_csv(f) for f in
     sorted((model_output / f"2-stage-XGBoost_global_adm1{RUN_SUFFIX}").glob("*.csv"))],
    ignore_index=True)
baseline = pd.read_parquet(model_output / "windspeed-exposed_global_adm1.parquet")
for frame in (loocv, baseline):
    for column in ADM1_KEYS:
        frame[column] = frame[column].astype(str)

observed = post_landfall(table, loocv, baseline)
print(f"post-landfall reference on {int(observed['units'].iloc[0]):,} units")
display(observed)

show = ["threshold", "lead_bin", "lead", "units", "events",
        "precision_xgb_median", "recall_xgb_median",
        "precision_wind_median", "recall_wind_median"]
metrics[show].round(3)

In [ ]:
def plot_skill_against_lead(metrics, observed, thresholds=(0, 15), savefig=None):
    """Precision and recall against time to landfall, per threshold.

    Rows are the two metrics, columns the two impact thresholds. Lead time
    runs down the x axis, so the panel reads the way the forecasts arrived:
    the earliest on the left, the storm landing on the right, and the
    detached point beyond it the score from the observed wind field.
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

    # One axis serves every panel, so the ticks and the reversal are set from
    # the whole frame rather than per threshold.
    leads = np.sort(metrics["lead"].unique())
    post = leads.min() - 14

    for column, threshold in enumerate(thresholds):
        band = metrics[metrics["threshold"] == threshold].sort_values("lead")

        for row, metric in enumerate(("precision", "recall")):
            ax = axes[row, column]
            for model in ("xgb", "wind"):
                middle = band[f"{metric}_{model}_median"]
                low = np.minimum.reduce([band[f"{metric}_{model}_q25"],
                                         band[f"{metric}_{model}_q75"], middle])
                high = np.maximum.reduce([band[f"{metric}_{model}_q25"],
                                          band[f"{metric}_{model}_q75"], middle])
                ax.plot(band["lead"], middle, color=COLOUR[model],
                        marker=MARKER[model], lw=2,
                        label=LABEL[model] if (row, column) == (0, 0) else None)
                ax.fill_between(band["lead"], low, high,
                                color=COLOUR[model], alpha=0.2)

                point = observed[(observed["model"] == model)
                                 & (observed["threshold"] == threshold)]
                if not point.empty:
                    ax.scatter(post, point[metric].iloc[0], color=COLOUR[model],
                               marker=MARKER[model], s=80, alpha=0.9,
                               zorder=3)

            ax.axvline((leads.min() + post) / 2, color="0.6", lw=0.8, ls=":")
            ax.set_ylim(0, 1)
            ax.grid(True, linestyle="--", alpha=0.6)
            ax.tick_params(labelsize=11)
            if row == 0:
                ax.set_title(f"Threshold = {threshold}%", fontsize=15)

    ticks = list(leads) + [post]
    labels = [f"{t:.0f}" for t in leads] + ["Observed"]
    for ax in axes.ravel():
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, rotation=45, fontsize=9)

    # Reversed, so time to landfall counts down to the storm's arrival. The
    # axis is shared, so setting it once sets all four panels.
    axes[0, 0].set_xlim(leads.max() + 8, post - 8)

    axes[0, 0].set_ylabel("Precision", fontsize=15)
    axes[1, 0].set_ylabel("Recall", fontsize=15)
    axes[1, 0].set_xlabel("Time to landfall (hours)", fontsize=15, x=1.05)
    axes[0, 0].legend(fontsize=11, loc="upper left")

    fig.tight_layout()
    if savefig:
        fig.savefig(savefig, dpi=300, bbox_inches="tight", pad_inches=0.2)
    return fig


plot_skill_against_lead(metrics, observed,
                        savefig=FIGDIR / "forecast_precision_recall.png")
plt.show()

## 3. One storm's ensemble, cycle by cycle

Each panel is a forecast cycle: the ensemble members' tracks over the ADM1
units as the model predicted them from that cycle. Panels run from the
earliest forecast to the latest, so the ensemble should tighten and the
predicted units settle as the storm approaches.

In [ ]:
import geopandas as gpd

from src.dynamic_features.process_forecast_tracks import (member_tracks,
                                                          read_event,
                                                          select_storm)
from src.evaluation.forecast_study import study_events

event = study_events(disnos={MAP_EVENT}).iloc[0]
cycles = read_event(MAP_EVENT, provider=PROVIDER)
storm = select_storm(cycles, event["Event Name"])
tracks = member_tracks(storm)

units = table[table["DisNo."] == MAP_EVENT].copy()
shapes = gpd.read_file(OUTPUT_DIR / "SHP" / "adm1_dissolved.gpkg")
shapes = shapes[shapes["GID_0"] == event["iso3"]][["GID_1", "geometry"]]
shapes["GID_1"] = shapes["GID_1"].astype(str)

print(f"{MAP_EVENT} ({event['Event Name']}): {len(tracks)} member tracks, "
      f"{units['cycle'].nunique()} cycles, {len(shapes)} ADM1 units")
print(f"lead times: "
      f"{sorted(units['lead_hours'].unique().round(0).astype(int), reverse=True)}")

In [ ]:
def plot_ensemble_map(tracks, shapes, units, column="xgb_median", ncols=4,
                      pad=1.0, savefig=None):
    """One panel per forecast cycle: the ensemble's tracks over its prediction.

    The colour scale is shared across panels so that a unit darkening between
    two panels means the forecast changed, not the scale.
    """
    order = units.sort_values("lead_hours", ascending=False)["cycle"].unique()
    by_cycle = {cycle: [t for t in tracks if t["cycle"].iloc[0] == cycle]
                for cycle in order}

    nrows = int(np.ceil(len(order) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    vmax = max(units[column].max(), 1e-6)
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)
    bounds = shapes.total_bounds

    for ax, cycle in zip(axes, order):
        panel = units[units["cycle"] == cycle]
        merged = shapes.merge(panel[["GID_1", column]], on="GID_1", how="left")
        merged.plot(ax=ax, column=column, cmap="OrRd", norm=norm,
                    edgecolor="black", linewidth=0.2, alpha=0.85,
                    missing_kwds={"color": "lightgray"})

        for track in by_cycle.get(cycle, []):
            ax.plot(track["longitude"], track["latitude"], color="darkgray",
                    lw=1, alpha=0.6, zorder=2)

        lead = panel["lead_hours"].iloc[0]
        ax.set_title(f"{lead:.0f} h to landfall", fontsize=13)
        ax.set_xlim(bounds[0] - pad, bounds[2] + pad)
        ax.set_ylim(bounds[1] - pad, bounds[3] + pad)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    bar = fig.add_axes([0.15, -0.02, 0.7, 0.025])
    mappable = mpl.cm.ScalarMappable(cmap="OrRd", norm=norm)
    mappable.set_array([])
    colourbar = fig.colorbar(mappable, cax=bar, orientation="horizontal")
    colourbar.set_label("Predicted population affected (%)", fontsize=13)
    colourbar.ax.tick_params(labelsize=11)

    if savefig:
        fig.savefig(savefig, dpi=300, bbox_inches="tight", pad_inches=0.2)
    return fig


plot_ensemble_map(tracks, shapes, units,
                  savefig=FIGDIR / "forecast_ensemble_map.png")
plt.show()

## 4. What the study rests on

Coverage is the limit worth stating with the result: the archive this is built
from reaches one basin, so the skill curve above describes that basin.

In [ ]:
covered = (table.groupby("DisNo.")
           .agg(cycles=("cycle", "nunique"), units=("GID_1", "nunique"),
                members=("members", "max"), lead=("lead_hours", "max"))
           .reset_index())
covered["iso3"] = covered["DisNo."].str.split("-").str[-1]

print(f"events          {len(covered)}")
print(f"countries       {covered['iso3'].nunique()}")
print(f"forecast cycles {int(covered['cycles'].sum())}")
print(f"cycles/event    median {covered['cycles'].median():.0f}, "
      f"max {int(covered['cycles'].max())}")
print(f"single-member   {int((covered['members'] < 2).sum())} events "
      f"(no ensemble spread; the band is undefined there)")

display(covered.groupby("iso3")
        .agg(events=("DisNo.", "nunique"), cycles=("cycles", "sum"))
        .sort_values("events", ascending=False))